# 1. The four subspaces of a linear map  [1.6–1.8.1]

TOML-MIRI — Linear Algebra for Data Science, exploration exercises.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.set_printoptions(precision=4, suppress=True)

---
## 1.1 See the four subspaces in 3-D

**Task.** Take a random $3\times 3$ matrix of rank 2. Compute bases of its row space, null space, column space and left null space with the SVD. Plot each pair (row space + null space, column space + left null space) in an interactive 3-D figure.

### Prediction (write this *before* running)

For a $3\times3$ matrix of rank 2, I expect:

- $\dim \mathrm{RS}(A) = \dots$, $\dim \mathrm{NS}(A) = \dots$ → geometrically: a ___ and a ___
- $\dim \mathrm{CS}(A) = \dots$, $\dim \mathrm{LNS}(A) = \dots$ → geometrically: a ___ and a ___
- The angle between the row space and the null space will be: ___

*(fill in)*

### Building a matrix with a chosen rank

A product $A = XY$ with $X\in\mathbb{R}^{m\times r}$, $Y\in\mathbb{R}^{r\times n}$ has rank at most $r$ (every column of $A$ is a combination of the $r$ columns of $X$). With random Gaussian $X, Y$ the rank is exactly $r$ with probability 1.

In [2]:
def random_matrix_of_rank(m, n, r, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((m, r))
    Y = rng.standard_normal((r, n))
    return X @ Y

A = random_matrix_of_rank(3, 3, 2, seed=0)
print(A)
print("rank:", np.linalg.matrix_rank(A))

[[ 0.3311  0.2014 -0.0939]
 [ 0.7024  0.5412 -0.4464]
 [-1.1561 -0.7327  0.3919]]
rank: 2


### The four subspaces from the SVD

With the full SVD $A = U\Sigma V^\top$ ($U \in \mathbb{R}^{m\times m}$, $V\in\mathbb{R}^{n\times n}$ orthogonal) and $r$ = number of non-zero singular values:

| subspace | lives in | basis | dimension |
|---|---|---|---|
| row space $\mathrm{RS}(A)$ | $\mathbb{R}^n$ | first $r$ columns of $V$ | $r$ |
| null space $\mathrm{NS}(A)$ | $\mathbb{R}^n$ | last $n-r$ columns of $V$ | $n-r$ |
| column space $\mathrm{CS}(A)$ | $\mathbb{R}^m$ | first $r$ columns of $U$ | $r$ |
| left null space $\mathrm{LNS}(A)$ | $\mathbb{R}^m$ | last $m-r$ columns of $U$ | $m-r$ |

"Non-zero" numerically means above a tolerance; we use the same one as `np.linalg.matrix_rank`.

In [3]:
def four_subspaces(A):
    m, n = A.shape
    U, s, Vt = np.linalg.svd(A)              # full SVD: U (m×m), Vt (n×n)
    tol = s.max() * max(m, n) * np.finfo(float).eps
    r = int(np.sum(s > tol))
    V = Vt.T
    return {
        "U": U, "s": s, "V": V, "r": r,
        "RS":  V[:, :r],    # in R^n
        "NS":  V[:, r:],    # in R^n
        "CS":  U[:, :r],    # in R^m
        "LNS": U[:, r:],    # in R^m
    }

S = four_subspaces(A)
print("singular values:", S["s"], " -> rank", S["r"])
for name in ["RS", "NS", "CS", "LNS"]:
    print(f"\n{name} basis (columns), dim = {S[name].shape[1]}:\n{S[name]}")

singular values: [1.7724 0.1728 0.    ]  -> rank 2

RS basis (columns), dim = 2:
[[-0.7844  0.4759]
 [-0.5261 -0.1705]
 [ 0.3287  0.8628]]

NS basis (columns), dim = 1:
[[-0.3978]
 [ 0.8332]
 [ 0.3841]]

CS basis (columns), dim = 2:
[[-0.2237  0.2442]
 [-0.5542 -0.8282]
 [ 0.8017 -0.5044]]

LNS basis (columns), dim = 1:
[[0.9436]
 [0.083 ]
 [0.3207]]


### Verify numerically (rule 3)

Each claim gets a one-line `np.allclose` check.

In [4]:
def verify(A, S):
    m, n = A.shape
    U, s, V, r = S["U"], S["s"], S["V"], S["r"]
    Sigma = np.zeros((m, n)); np.fill_diagonal(Sigma, s)
    checks = {
        "A = U Σ Vᵀ":                       np.allclose(A, U @ Sigma @ V.T),
        "A · NS = 0      (NS ⊂ ker A)":     np.allclose(A @ S["NS"], 0),
        "Aᵀ · LNS = 0    (LNS ⊂ ker Aᵀ)":   np.allclose(A.T @ S["LNS"], 0),
        "RSᵀ · NS = 0    (RS ⟂ NS)":        np.allclose(S["RS"].T @ S["NS"], 0),
        "CSᵀ · LNS = 0   (CS ⟂ LNS)":       np.allclose(S["CS"].T @ S["LNS"], 0),
        "rows of A lie in RS":              np.allclose(S["RS"] @ S["RS"].T @ A.T, A.T),
        "columns of A lie in CS":           np.allclose(S["CS"] @ S["CS"].T @ A, A),
        f"dim RS + dim NS = n = {n}":       S["RS"].shape[1] + S["NS"].shape[1] == n,
        f"dim CS + dim LNS = m = {m}":      S["CS"].shape[1] + S["LNS"].shape[1] == m,
    }
    for k, v in checks.items():
        print(f"{'✓' if v else '✗'}  {k}")

verify(A, S)

✓  A = U Σ Vᵀ
✓  A · NS = 0      (NS ⊂ ker A)
✓  Aᵀ · LNS = 0    (LNS ⊂ ker Aᵀ)
✓  RSᵀ · NS = 0    (RS ⟂ NS)
✓  CSᵀ · LNS = 0   (CS ⟂ LNS)
✓  rows of A lie in RS
✓  columns of A lie in CS
✓  dim RS + dim NS = n = 3
✓  dim CS + dim LNS = m = 3


### Plotting helpers

A subspace of $\mathbb{R}^3$ is drawn according to its dimension:

- dim 0 → just the origin (a dot)
- dim 1 → a line through the origin
- dim 2 → a plane through the origin
- dim 3 → all of $\mathbb{R}^3$ (a faint cube)

Basis vectors are drawn as arrows. On top we also plot the **rows** of $A$ (left panel) and the **columns** of $A$ (right panel) as markers, to see that they really lie in RS and CS.

In [5]:
L = 1.5  # half-size of the drawn region

def arrow(v, color, name, showlegend=False):
    return [
        go.Scatter3d(x=[0, v[0]], y=[0, v[1]], z=[0, v[2]], mode="lines",
                     line=dict(color=color, width=6), name=name,
                     legendgroup=name, showlegend=showlegend),
        go.Cone(x=[v[0]], y=[v[1]], z=[v[2]], u=[v[0]], v=[v[1]], w=[v[2]],
                sizemode="absolute", sizeref=0.15, anchor="tip",
                colorscale=[[0, color], [1, color]], showscale=False,
                legendgroup=name, showlegend=False, hoverinfo="skip"),
    ]

def subspace_traces(B, color, name):
    # Traces drawing span(columns of B) in R^3; B has orthonormal columns.
    d = B.shape[1]
    t = np.linspace(-L, L, 2)
    if d == 0:
        traces = [go.Scatter3d(x=[0], y=[0], z=[0], mode="markers",
                               marker=dict(size=8, color=color),
                               name=f"{name} = {{0}}", legendgroup=name)]
    elif d == 1:
        p = np.outer(B[:, 0], t)
        traces = [go.Scatter3d(x=p[0], y=p[1], z=p[2], mode="lines",
                               line=dict(color=color, width=4, dash="dash"),
                               name=f"{name} (line)", legendgroup=name)]
    elif d == 2:
        s_, t_ = np.meshgrid(t, t)
        P = B[:, [0]] * s_.ravel() + B[:, [1]] * t_.ravel()
        traces = [go.Surface(x=P[0].reshape(2, 2), y=P[1].reshape(2, 2), z=P[2].reshape(2, 2),
                             colorscale=[[0, color], [1, color]], opacity=0.35,
                             showscale=False, name=f"{name} (plane)",
                             legendgroup=name, showlegend=True)]
    else:  # d == 3: the whole space
        c = np.array([[x, y, z] for x in (-L, L) for y in (-L, L) for z in (-L, L)])
        traces = [go.Mesh3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], alphahull=0,
                            color=color, opacity=0.08, name=f"{name} = R³",
                            legendgroup=name, showlegend=True)]
    for i in range(d):
        traces += arrow(B[:, i], color, name)
    return traces

def points_trace(P, color, name):
    # P: 3 × k, each column a point
    return go.Scatter3d(x=P[0], y=P[1], z=P[2], mode="markers",
                        marker=dict(size=5, color=color, symbol="diamond",
                                    line=dict(color="black", width=1)),
                        name=name)

def plot_four_subspaces(A, title=""):
    S = four_subspaces(A)
    scale = L / np.abs(A).max()   # shrink rows/cols of A to fit the box
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                        subplot_titles=[f"Domain ℝⁿ: RS (dim {S['RS'].shape[1]}) ⟂ NS (dim {S['NS'].shape[1]})",
                                        f"Codomain ℝᵐ: CS (dim {S['CS'].shape[1]}) ⟂ LNS (dim {S['LNS'].shape[1]})"])
    for tr in subspace_traces(S["RS"], "royalblue", "RS") + subspace_traces(S["NS"], "crimson", "NS"):
        fig.add_trace(tr, row=1, col=1)
    fig.add_trace(points_trace(scale * A.T, "royalblue", "rows of A (scaled)"), row=1, col=1)
    for tr in subspace_traces(S["CS"], "seagreen", "CS") + subspace_traces(S["LNS"], "darkorange", "LNS"):
        fig.add_trace(tr, row=1, col=2)
    fig.add_trace(points_trace(scale * A, "seagreen", "columns of A (scaled)"), row=1, col=2)
    axis = dict(range=[-L, L])
    scene = dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube")
    fig.update_layout(scene=scene, scene2=scene, height=550,
                      title=title or f"rank(A) = {S['r']}", legend=dict(itemsizing="constant"))
    fig.show()
    return S

In [6]:
S = plot_four_subspaces(A, "Random 3×3 matrix of rank 2")

**Things to do with the figure:** rotate the left panel until you look *along* the null-space line — the row-space plane should collapse to a line perpendicular to it. Do the same on the right with the left null space. Check that the diamonds (rows / columns of $A$) sit exactly on the planes.

---
### Explore: change the rank to 1 and to 3

Before running: which subspaces do you expect to **grow**, **shrink** or **disappear** (become $\{0\}$)?

*(fill in)*

In [ ]:
for r in [1, 3]:
    A_r = random_matrix_of_rank(3, 3, r, seed=1)
    print(f"===== rank {r} =====")
    S_r = plot_four_subspaces(A_r, f"Random 3×3 matrix of rank {r}")
    verify(A_r, S_r)
    print()

Optional: an interactive slider over the rank (needs `ipywidgets`) and over the random seed.

In [ ]:
from ipywidgets import interact, IntSlider

@interact(rank=IntSlider(min=0, max=3, value=2), seed=IntSlider(min=0, max=20, value=0))
def explore(rank, seed):
    if rank == 0:
        print("rank 0: A = 0 (SVD-based plot skipped; what are the four subspaces here?)")
        return
    plot_four_subspaces(random_matrix_of_rank(3, 3, rank, seed), f"rank {rank}, seed {seed}")

### Dimension count for every case

In [ ]:
m = n = 3
print(f"{'rank':>4} | {'dim RS':>6} {'dim NS':>6} {'sum':>4} (n={n}) | {'dim CS':>6} {'dim LNS':>7} {'sum':>4} (m={m})")
for r in [1, 2, 3]:
    S_r = four_subspaces(random_matrix_of_rank(m, n, r, seed=r))
    dRS, dNS, dCS, dLNS = (S_r[k].shape[1] for k in ["RS", "NS", "CS", "LNS"])
    print(f"{S_r['r']:>4} | {dRS:>6} {dNS:>6} {dRS + dNS:>4}       | {dCS:>6} {dLNS:>7} {dCS + dLNS:>4}")
    assert dRS + dNS == n and dCS + dLNS == m

---
### Explain (hand-in — in your own words)

**1. Which subspaces grow, shrink or disappear as the rank goes 1 → 2 → 3?**

*Your answer.*

**2. Why is the null space always perpendicular to the row space?**
*(Hint: write out what $Ax = 0$ says row by row, and what that means for $x$ and every row $a_i^\top$ — and hence every combination of rows.)*

*Your answer.*

**3. Check $\dim \mathrm{RS}(A) + \dim \mathrm{NS}(A) = n$ and $\dim \mathrm{CS}(A) + \dim \mathrm{LNS}(A) = m$ for each case. Why must these hold?**
*(Hint: the columns of $V$ are an orthonormal basis of $\mathbb{R}^n$, split into the first $r$ and the last $n-r$.)*

*Your answer.*